# Tutorial: Optimization with SciPy and Dimensionality Reduction with PCA
## DA5401W - Foundations of Machine Learning
### Instructor: Dr. Arun B Ayyar
### IIT Madras

---

## Instructions

This tutorial contains **hands-on problems** for:
1. **Optimization using SciPy** (Linear Programming, Nonlinear Optimization, Constrained Optimization)
2. **Dimensionality Reduction using PCA** (Feature extraction, visualization, reconstruction)

**How to use this notebook:**
- Each problem has a **Problem Statement** with data setup
- Try solving the problem yourself first
- Solutions will be provided by the TA
- Run all cells to load the data and helper functions

**Learning Objectives:**
- Apply scipy.optimize for real-world optimization problems
- Understand when to use different optimization methods
- Implement PCA for dimensionality reduction
- Interpret principal components and explained variance
- Apply PCA to real datasets

In [4]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize, linprog, LinearConstraint, NonlinearConstraint
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer, load_wine, fetch_olivetti_faces

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Set random seed
np.random.seed(42)

print("✓ All libraries imported successfully!")
print("✓ Ready to start the tutorial")

✓ All libraries imported successfully!
✓ Ready to start the tutorial


---
# Part 1: Optimization with SciPy

## Problem 1: Production Planning (Linear Programming)

### Problem Statement

A factory produces two products: **Product A** and **Product B**.

**Profit:**
- Product A: ₹50 per unit
- Product B: ₹40 per unit

**Resource Constraints:**
- **Labor hours**: Product A needs 2 hours, Product B needs 1 hour. Total available: 100 hours
- **Raw material**: Product A needs 1 kg, Product B needs 2 kg. Total available: 80 kg
- **Machine time**: Product A needs 1 hour, Product B needs 1 hour. Total available: 60 hours

**Question:** How many units of each product should be produced to maximize profit?

**Mathematical Formulation:**

Maximize: $Z = 50x_1 + 40x_2$

Subject to:
- $2x_1 + x_2 \leq 100$ (labor)
- $x_1 + 2x_2 \leq 80$ (raw material)
- $x_1 + x_2 \leq 60$ (machine time)
- $x_1, x_2 \geq 0$

In [5]:
# Data for Problem 1
# Objective function coefficients (we need to minimize, so negate for maximization)
c = [-50, -40]  # Negative because linprog minimizes

# Inequality constraint matrix (A_ub @ x <= b_ub)
A_ub = np.array(
    [
        [2, 1],  # Labor constraint
        [1, 2],  # Raw material constraint
        [1, 1],  # Machine time constraint
    ]
)

b_ub = np.array([100, 80, 60])

# Bounds for variables (x1 >= 0, x2 >= 0)
bounds = [(0, None), (0, None)]

print("Problem 1 Data Loaded")
print("=" * 60)
print("Objective: Maximize 50*x1 + 40*x2")
print("\nConstraints:")
print("  2*x1 + 1*x2 <= 100  (Labor)")
print("  1*x1 + 2*x2 <= 80   (Raw material)")
print("  1*x1 + 1*x2 <= 60   (Machine time)")
print("  x1, x2 >= 0")
print("\n→ YOUR TASK: Use scipy.optimize.linprog to solve this problem")

Problem 1 Data Loaded
Objective: Maximize 50*x1 + 40*x2

Constraints:
  2*x1 + 1*x2 <= 100  (Labor)
  1*x1 + 2*x2 <= 80   (Raw material)
  1*x1 + 1*x2 <= 60   (Machine time)
  x1, x2 >= 0

→ YOUR TASK: Use scipy.optimize.linprog to solve this problem


In [6]:
# YOUR SOLUTION HERE
# Use: result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

result

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -2800.0
              x: [ 4.000e+01  2.000e+01]
            nit: 2
          lower:  residual: [ 4.000e+01  2.000e+01]
                 marginals: [ 0.000e+00  0.000e+00]
          upper:  residual: [       inf        inf]
                 marginals: [ 0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 0.000e+00  0.000e+00  0.000e+00]
                 marginals: [-2.000e+01 -1.000e+01 -0.000e+00]
 mip_node_count: 0
 mip_dual_bound: 0.0
        mip_gap: 0.0

In [8]:
profit = -result['fun']
product_a_unit = result['x'][0]
product_b_unit = result['x'][1]
labor_cost = 2*product_a_unit + product_b_unit
raw_material = product_a_unit + 2*product_b_unit
machine_time = product_a_unit + product_b_unit

In [9]:
print(f"Profit: {profit}\
       No. of Product A units: {product_a_unit} \
       No. of Product B units: {product_b_unit} \
       Labor Cost: {labor_cost} \
       Raw Material: {raw_material} \
       Machine Time: {machine_time} \
      ")

Profit: 2800.0       No. of Product A units: 40.0        No. of Product B units: 20.0        Labor Cost: 100.0        Raw Material: 80.0        Machine Time: 60.0       


Problem 1.2: Nutrient Optimization (Cost Minimization)
Problem Statement
A fitness center wants to create a supplement mix using two ingredients: Ingredient X and Ingredient Y.

Cost:

Ingredient X: ₹30 per unit

Ingredient Y: ₹50 per unit

Nutritional Requirements: To ensure the mix is effective, it must meet the following minimum daily requirements:

Protein: Ingredient X provides 3 units, Ingredient Y provides 2 units. Requirement: At least 12 units

Vitamins: Ingredient X provides 1 unit, Ingredient Y provides 3 units. Requirement: At least 9 units

Carbohydrates: Ingredient X provides 1 unit, Ingredient Y provides 1 unit. Requirement: At least 6 units

Question: How many units of Ingredient X and Ingredient Y should be purchased to minimize the total cost while meeting all nutritional requirements?

Mathematical Formulation:

Objective: minimize(30X + 50Y)

Subject to:
3X + 2Y >= 12 => -3X - 2Y <= -12
X + 3Y >= 9   => -X - 3Y <= -9
X + Y >= 6    => -X - Y <= -6
X, Y >=0

In [14]:
c = [30, 50]

st_c = np.array([
    [-3, -2],
    [-1, -3],
    [-1, -1]
]) 

st_b = np.array([-12, -9, -6])

bounds = [(0, None),(0, None)]

result = linprog(c, A_ub=st_c, b_ub=st_b, bounds=bounds, method="highs")
result

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: 210.0
              x: [ 4.500e+00  1.500e+00]
            nit: 3
          lower:  residual: [ 4.500e+00  1.500e+00]
                 marginals: [ 0.000e+00  0.000e+00]
          upper:  residual: [       inf        inf]
                 marginals: [ 0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 4.500e+00  0.000e+00  0.000e+00]
                 marginals: [-0.000e+00 -1.000e+01 -2.000e+01]
 mip_node_count: 0
 mip_dual_bound: 0.0
        mip_gap: 0.0

In [18]:
print(f"No of X: {result['x'][0]:.2f}   No. of Y: {result['x'][1]:.2f} Min Nutrient is: {result['fun']:.2f}")

No of X: 4.50   No. of Y: 1.50 Min Nutrient is: 210.00


---
## Problem 2: Portfolio Optimization (Quadratic Programming)

### Problem Statement

You want to invest in 3 stocks. Historical data shows:

**Expected Returns:**
- Stock 1: 12% per year
- Stock 2: 10% per year
- Stock 3: 8% per year

**Covariance Matrix** (risk):
```
        Stock1  Stock2  Stock3
Stock1   0.04    0.01    0.00
Stock2   0.01    0.03    0.01
Stock3   0.00    0.01    0.02
```

**Question:** Find the portfolio weights that **minimize risk** (variance) while achieving **at least 10% expected return**.

**Mathematical Formulation:**

Minimize: $\frac{1}{2}w^T \Sigma w$ (portfolio variance)

Subject to:
- $\mu^T w \geq 0.10$ (minimum return)
- $\sum w_i = 1$ (fully invested)
- $w_i \geq 0$ (no short selling)

In [ ]:
# Data for Problem 2
# Expected returns
mu = np.array([0.12, 0.10, 0.08])

# Covariance matrix
Sigma = np.array([[0.04, 0.01, 0.00], [0.01, 0.03, 0.01], [0.00, 0.01, 0.02]])

print("Problem 2 Data Loaded")
print("=" * 60)
print("Expected Returns:", mu)
print("\nCovariance Matrix:")
print(Sigma)
print("\n→ YOUR TASK: Find optimal portfolio weights")
print("   - Minimize portfolio variance")
print("   - Expected return >= 10%")
print("   - Weights sum to 1")
print("   - No short selling (w >= 0)")

In [ ]:
# YOUR SOLUTION HERE
# Hint: Define objective function as: lambda w: 0.5 * w @ Sigma @ w
# Use LinearConstraint and NonlinearConstraint for constraints
from scipy.optimize import LinearConstraint, minimize

def portfolio_variance(w, Sigma):
    return 0.5 * w.T @ Sigma @ w

def portfolio_variance_derivative(w, Sigma):
    return w @ Sigma

constraint_sum = LinearConstraint(np.ones(3), 1, 1)

constraint_return = LinearConstraint(mu, 0.1, np.inf)


---
## Problem 3: Rosenbrock Function Minimization

### Problem Statement

The **Rosenbrock function** is a famous test function for optimization algorithms:

$$f(x, y) = (1-x)^2 + 100(y-x^2)^2$$

It has a narrow parabolic valley with the global minimum at $(1, 1)$ where $f(1, 1) = 0$.

**Question:** 
1. Minimize the Rosenbrock function starting from $(-1.5, 2.5)$
2. Compare different optimization methods: `'BFGS'`, `'Nelder-Mead'`, `'CG'`
3. Which method converges fastest?

In [ ]:
# Data for Problem 3
def rosenbrock(x):
    """Rosenbrock function"""
    return (1 - x[0]) ** 2 + 100 * (x[1] - x[0] ** 2) ** 2


def rosenbrock_grad(x):
    """Gradient of Rosenbrock function"""
    dfdx = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0] ** 2)
    dfdy = 200 * (x[1] - x[0] ** 2)
    return np.array([dfdx, dfdy])


# Starting point
x0 = np.array([-1.5, 2.5])

print("Problem 3 Data Loaded")
print("=" * 60)
print("Function: f(x,y) = (1-x)² + 100(y-x²)²")
print(f"Starting point: {x0}")
print(f"Function value at start: {rosenbrock(x0):.2f}")
print("\n→ YOUR TASK: Minimize using different methods and compare")

In [ ]:
# YOUR SOLUTION HERE
# Try methods: 'BFGS', 'Nelder-Mead', 'CG'

---
## Problem 4: Breast Cancer Dataset - Feature Extraction

### Problem Statement

The **Breast Cancer Wisconsin dataset** contains 569 samples of breast cancer tumors with 30 features computed from digitized images:
- Features include: radius, texture, perimeter, area, smoothness, compactness, concavity, etc.
- Each feature is computed as mean, standard error, and "worst" (mean of 3 largest values)
- Examples: mean radius, radius error, worst radius

There are 2 classes: **Malignant** (cancerous) and **Benign** (non-cancerous).

**Questions:**
1. Apply PCA to reduce from 30D to 2D
2. How much variance is explained by the first 2 principal components?
3. Visualize the data in the 2D PCA space
4. Which original features contribute most to PC1 and PC2?
5. Are the two classes (malignant vs benign) separable in 2D?

In [ ]:
# Data for Problem 4
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target
feature_names_cancer = cancer.feature_names
target_names_cancer = cancer.target_names

print("Problem 4 Data Loaded: Breast Cancer Dataset")
print("=" * 60)
print(f"Number of samples: {X_cancer.shape[0]}")
print(f"Number of features: {X_cancer.shape[1]}")
print(f"\nFirst 5 feature names:")
for i, name in enumerate(feature_names_cancer[:5]):
    print(f"  {i+1}. {name}")
print(f"  ... and {len(feature_names_cancer)-5} more features")
print(f"\nTarget names: {target_names_cancer}")
print(f"  - Malignant: {np.sum(y_cancer == 0)} samples")
print(f"  - Benign: {np.sum(y_cancer == 1)} samples")
print(f"\nFirst 5 samples (first 5 features):")
print(pd.DataFrame(X_cancer[:5, :5], columns=feature_names_cancer[:5]))
print("\n→ YOUR TASK: Apply PCA and answer the questions above")

In [ ]:
# YOUR SOLUTION HERE
# Steps:
# 1. Standardize the data using StandardScaler
# 2. Apply PCA with n_components=2
# 3. Transform the data
# 4. Analyze explained variance
# 5. Visualize

---
## Problem 5: Olivetti Faces - Image Reconstruction

### Problem Statement

The **Olivetti Faces dataset** contains 400 grayscale face images of 40 different people (10 images per person). Each image is 64×64 pixels = 4096 features.

**Questions:**
1. Apply PCA and create a **scree plot** to determine how many components to keep
2. How many components are needed to retain 90% of variance?
3. Reduce to that many components and reconstruct the images
4. Visualize original vs reconstructed images
5. What is the compression ratio achieved?

In [ ]:
# Data for Problem 5
faces = fetch_olivetti_faces()
X_faces = faces.data
y_faces = faces.target

print("Problem 5 Data Loaded: Olivetti Faces")
print("=" * 60)
print(f"Number of samples: {X_faces.shape[0]}")
print(f"Number of features (pixels): {X_faces.shape[1]}")
print(f"Image shape: 64x64")
print(f"Number of people: {len(np.unique(y_faces))}")

# Show some example images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_faces[i].reshape(64, 64), cmap="gray")
    ax.set_title(f"Person {y_faces[i]}")
    ax.axis("off")
plt.suptitle("Sample Face Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n→ YOUR TASK: Apply PCA for image compression and reconstruction")

In [ ]:
# YOUR SOLUTION HERE
# Steps:
# 1. Standardize the data
# 2. Apply PCA with all components first
# 3. Create scree plot
# 4. Find number of components for 90% variance
# 5. Reconstruct images

---
## Problem 6: Wine Dataset - Feature Importance

### Problem Statement

The **Wine dataset** contains chemical analysis of 178 wines with 13 features:
- Alcohol, Malic acid, Ash, Alkalinity of ash, Magnesium, Total phenols, Flavanoids, 
  Nonflavanoid phenols, Proanthocyanins, Color intensity, Hue, OD280/OD315, Proline

There are 3 wine cultivars (classes).

**Questions:**
1. Apply PCA and reduce to 2D
2. Which original features are most important for PC1?
3. Which original features are most important for PC2?
4. Create a biplot showing both data points and feature loadings
5. Can you identify which features distinguish the wine cultivars?

In [ ]:
# Data for Problem 6
wine = load_wine()
X_wine = wine.data
y_wine = wine.target
feature_names_wine = wine.feature_names
target_names_wine = wine.target_names

print("Problem 6 Data Loaded: Wine Dataset")
print("=" * 60)
print(f"Number of samples: {X_wine.shape[0]}")
print(f"Number of features: {X_wine.shape[1]}")
print(f"\nFeatures:")
for i, name in enumerate(feature_names_wine, 1):
    print(f"  {i}. {name}")
print(f"\nTarget names: {target_names_wine}")
print("\n→ YOUR TASK: Apply PCA and analyze feature importance")

In [ ]:
# YOUR SOLUTION HERE
# Steps:
# 1. Standardize
# 2. Apply PCA with 2 components
# 3. Analyze loadings
# 4. Create biplot

---
## Summary and Key Takeaways

### Optimization with SciPy

1. **Linear Programming** (`linprog`):
   - For problems with linear objective and linear constraints
   - Use `method='highs'` for best performance
   - Remember to negate coefficients when maximizing

2. **Nonlinear Optimization** (`minimize`):
   - Choose method based on problem characteristics:
     - `'BFGS'`: Fast, requires gradient (can approximate)
     - `'Nelder-Mead'`: No gradient needed, robust but slower
     - `'SLSQP'`: For constrained problems
   - Always provide good initial guesses
   - Use analytical gradients when possible for speed

3. **Constraints**:
   - Use `LinearConstraint` for linear constraints
   - Use `NonlinearConstraint` for nonlinear constraints
   - Bounds are specified separately

### PCA for Dimensionality Reduction

1. **Always standardize** data before PCA (using `StandardScaler`)

2. **Choosing number of components**:
   - Use scree plot (elbow method)
   - Cumulative variance threshold (e.g., 90%, 95%)
   - Cross-validation for downstream tasks

3. **Interpretation**:
   - Explained variance ratio: how much information each PC captures
   - Component loadings: which original features contribute to each PC
   - Biplots: visualize both data and feature relationships

4. **Applications**:
   - Visualization (reduce to 2D or 3D)
   - Feature extraction for machine learning
   - Noise reduction
   - Data compression

### Best Practices

- **Optimization**: Start simple, validate results, check convergence
- **PCA**: Understand what variance means in your domain
- **Both**: Always visualize your results!

---
